# ChemiAI — предсказание противовирусной активности соединений

Ноутбук с решением задачи регрессии по молекулярным дескрипторам: для каждого соединения предсказываются **IC50**, **CC50** и **SI**.

## Описание задачи

Разработка лекарств — длительный процесс синтеза и биологического тестирования. Машинное обучение позволяет оценивать перспективность соединений **до** лабораторных экспериментов.

В датасете — свойства молекул и их активность против вируса гриппа. Нужно построить модель, предсказывающую для новых соединений:

| Таргет | Смысл |
|--------|--------|
| **IC50 (mM)** | концентрация, при которой подавляется 50% активности вируса |
| **CC50 (mM)** | концентрация токсичности для 50% клеток |
| **SI** | индекс селективности (Selectivity Index) |

**Метрика:** среднее RMSE по трём таргетам:

$$\text{score} = \frac{RMSE(IC50) + RMSE(CC50) + RMSE(SI)}{3}$$

**Формат сабмита:** `index, IC50, CC50, SI` (см. `sample_submission.csv`).

Подробнее — в [DESCRIPTION.md](./DESCRIPTION.md). Данные: [Google Drive](https://drive.google.com/drive/folders/1m1PS44rF9HqIAOUZQeopU5sqwgi6YfO8?usp=sharing).

## План работы

1. **Загрузка данных** — скачивание с Google Drive и чтение `train` / `test` / `sample_submission`.
2. **EDA** — распределения таргетов и признаков, пропуски, корреляции.
3. **Предобработка** — масштабирование, отбор признаков при необходимости.
4. **Моделирование** — baseline и улучшенные модели (multi-target или отдельные регрессоры).
5. **Валидация** — кросс-валидация по метрике соревнования.
6. **Сабмит** — предсказания на `test.csv` и сохранение `submission.csv`.

## 1. Настройка окружения

In [12]:
import gdown
from pathlib import Path

import numpy as np
import pandas as pd

# Воспроизводимость
RANDOM_STATE = 42
np.random.seed(RANDOM_STATE)

# Пути
ROOT = Path(".").resolve()
DATA_DIR = ROOT / "data"
GDRIVE_FOLDER_ID = "1m1PS44rF9HqIAOUZQeopU5sqwgi6YfO8"

TRAIN_PATH = DATA_DIR / "train.csv"
TEST_PATH = DATA_DIR / "test.csv"
SAMPLE_SUBMISSION_PATH = DATA_DIR / "sample_submission.csv"

## 2. Загрузка данных

Датасет: [Google Drive](https://drive.google.com/drive/folders/1m1PS44rF9HqIAOUZQeopU5sqwgi6YfO8?usp=sharing) (`train.csv`, `test.csv`, `sample_submission.csv`).

In [13]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
gdown.download_folder(id=GDRIVE_FOLDER_ID, output=str(DATA_DIR))

Retrieving folder contents


Processing file 1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH sample_submission.csv
Processing file 1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l test.csv
Processing file 159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl train.csv


Retrieving folder contents completed
Building directory structure
Building directory structure completed
Downloading...
From: https://drive.google.com/uc?id=1LL6moSzpUVxJUTMeXihWvUxBJNjvj6EH
To: /Users/dmitriytonkusin/Mephi/Машинное обучение без учителя/mephi-unsupervised-learning/ChemiAI/data/sample_submission.csv
100%|██████████| 15.4k/15.4k [00:00<00:00, 295kB/s]
Downloading...
From: https://drive.google.com/uc?id=1Ui2t87X3in-Wu-pnjkDXa_VtPsVafi0l
To: /Users/dmitriytonkusin/Mephi/Машинное обучение без учителя/mephi-unsupervised-learning/ChemiAI/data/test.csv
100%|██████████| 441k/441k [00:00<00:00, 785kB/s]
Downloading...
From: https://drive.google.com/uc?id=159PZX3X5rpUO-WbzWyC9whnc8B4mNqJl
To: /Users/dmitriytonkusin/Mephi/Машинное обучение без учителя/mephi-unsupervised-learning/ChemiAI/data/train.csv
100%|██████████| 1.36M/1.36M [00:00<00:00, 2.01MB/s]
Download completed


['/Users/dmitriytonkusin/Mephi/Машинное обучение без учителя/mephi-unsupervised-learning/ChemiAI/data/sample_submission.csv',
 '/Users/dmitriytonkusin/Mephi/Машинное обучение без учителя/mephi-unsupervised-learning/ChemiAI/data/test.csv',
 '/Users/dmitriytonkusin/Mephi/Машинное обучение без учителя/mephi-unsupervised-learning/ChemiAI/data/train.csv']

In [14]:
train = pd.read_csv(TRAIN_PATH)
test = pd.read_csv(TEST_PATH)
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print(f"Train: {train.shape}")
print(f"Test:  {test.shape}")
print(f"Sample submission: {sample_submission.shape}")
train.head()

Train: (751, 214)
Test:  (250, 211)
Sample submission: (250, 4)


,index,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
0,0,102.414420,95.757483,0.935000,5.466584,5.466584,0.719259,0.719259,0.681165,18.307692,...,1,0,0,0,0,0,0,0,0,0
1,1,0.044333,8.401080,189.500000,11.492712,11.492712,0.012350,-3.798024,0.769122,27.652174,...,0,1,0,0,0,0,0,0,0,0
2,2,4.437964,50.085589,11.285714,5.366084,5.366084,0.522930,0.522930,0.612606,24.608696,...,0,0,0,0,0,0,0,0,0,0
3,3,6.827881,682.788051,100.000000,13.317130,13.317130,0.020658,-4.829339,0.345823,12.400000,...,0,0,1,0,0,0,0,0,0,0
4,4,2.003253,70.001455,34.943894,6.320833,6.320833,0.300347,0.300347,0.562066,60.272727,...,0,0,0,0,0,0,0,0,0,0


### Первичный обзор

Проверяем структуру таблиц, пропуски и согласованность признаков между train и test.

In [15]:
train.describe()

,index,"IC50, mM","CC50, mM",SI,MaxAbsEStateIndex,MaxEStateIndex,MinAbsEStateIndex,MinEStateIndex,qed,SPS,...,fr_sulfide,fr_sulfonamd,fr_sulfone,fr_term_acetylene,fr_tetrazole,fr_thiazole,fr_thiocyan,fr_thiophene,fr_unbrch_alkane,fr_urea
count,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,...,751.000000,751.000000,751.000000,751.000000,751.000000,751.000000,751.0,751.000000,751.000000,751.000000
mean,375.000000,204.544021,577.426098,89.153313,10.860070,10.860070,0.180064,-0.971890,0.577938,29.588010,...,0.055925,0.013316,0.010652,0.001332,0.001332,0.054594,0.0,0.069241,0.182423,0.006658
std,216.939316,370.367937,641.515163,788.882198,3.347314,3.347314,0.169163,1.594491,0.214190,12.713195,...,0.272400,0.114699,0.102728,0.036491,0.036491,0.227337,0.0,0.254033,1.227468,0.081377
min,0.000000,0.003517,0.700808,0.011489,2.321942,2.321942,0.000039,-6.992796,0.059567,9.545455,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
25%,187.500000,13.222351,99.998894,1.500000,8.921032,8.921032,0.048473,-1.333831,0.442842,18.306020,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
50%,375.000000,44.069306,376.580899,4.000000,12.197500,12.197500,0.121372,-0.419485,0.636477,29.281250,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
75%,562.500000,206.787402,877.508784,17.372463,13.214245,13.214245,0.290990,0.072488,0.742483,38.875000,...,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.0,0.000000,0.000000,0.000000
max,750.000000,4095.188563,4538.976189,15620.600000,15.933463,15.933463,1.374614,1.374614,0.947265,60.272727,...,2.000000,1.000000,1.000000,1.000000,1.000000,1.000000,0.0,1.000000,20.000000,1.000000
